# IPN 13 lớp: sweep cải thiện nhánh neural (chọn theo validation)

Mục tiêu: tăng chỉ số của neural-only baseline C bằng các thay đổi đã có lý do từ phân tích lỗi test của lần chạy trước, **chọn bằng validation, nhiều seed**, không nhìn test để chọn.

Lỗi lớn nhất của C nằm ở các cặp đếm click (G01/G08, G02/G09) và chiều pinch (G10/G11), và khác nhau theo camera. Bốn biến thể:

| Biến thể | Thay đổi so với C | Giả thuyết |
|---|---|---|
| `c_ref` | không | Mốc đối chứng, cùng code và cùng seed |
| `hand` | `--hand-features` | Khoảng cách ngón tay chuẩn hóa theo cỡ bàn tay cho tín hiệu click và pinch sạch hơn tọa độ chuẩn hóa theo vai |
| `aug` | `--time-crop-min 0.7 --scale-jitter 0.1 --shift-std 0.03` | Đa dạng tốc độ, vị trí, độ zoom giữa các subject và camera, không che frame nên không phá nhịp click |
| `hand_aug` | cả hai | Kết hợp |

Quy tắc: biến thể thắng = macro-F1 validation trung bình qua các seed cao nhất. Test mặc định **khóa** (`RUN_LOCKED_TEST=False`). Test IPN đã bị mở một lần ở lần chạy trước, nên nếu bật thì đây là lần xem thứ hai và chỉ là bằng chứng exploratory.

In [ ]:
from pathlib import Path
import csv
import datetime as dt
import hashlib
import json
import os
import shutil
import statistics
import subprocess
import sys
import uuid
import zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không chạy trên Colab: bỏ qua mount Drive.')

# --- Chỉ sửa các biến này ---
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_REF = 'HuyenHoang'      # nhánh chứa code mới (hand_geometry.py, ensemble_neural_only.py)
REPO_COMMIT = None           # đặt hash để ghim chính xác; None thì dùng đầu nhánh và ghi lại hash đã dùng
SEEDS = [42, 123, 3407]
RUN_LOCKED_TEST = False      # True = xem test lần 2 cho biến thể thắng và c_ref (exploratory)
NUM_CLASSES = 13
EXPECTED_COUNTS = {'train': 2443, 'val': 589, 'test': 1101}   # None để bỏ kiểm tra khi chạy thử nhỏ
DEVICE = 'cuda:0'

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:10]
WORK_ROOT = Path('/content') / f'ipn13_neural_sweep_{RUN_ID}'
REPO_DIR = WORK_ROOT / 'repo'
OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_neural_sweep'        # checkpoint từng run, giữ qua reconnect
DRIVE_RUN_ROOT = DRIVE_DATA_ROOT / 'runs' / 'ipn13_neural_sweep'
TRAIN_FILE = DRIVE_DATA_ROOT / 'train.txt'
VAL_FILE = DRIVE_DATA_ROOT / 'val.txt'
TEST_FILE = DRIVE_DATA_ROOT / 'test.txt'
print({'run_id': RUN_ID, 'seeds': SEEDS, 'locked_test': RUN_LOCKED_TEST})


## 1. Clone code và kiểm tra code mới đã được push

Code mới (đặc trưng hình học bàn tay, augmentation thời gian, ensemble) phải có trên nhánh. Nếu thiếu file, notebook dừng và nói rõ.

In [ ]:
COMMANDS = []
WORK_ROOT.mkdir(parents=True, exist_ok=True)


def run_argv(argv, cwd=None, log_path=None):
    command = [str(value) for value in argv]
    COMMANDS.append(command)
    print('$', ' '.join(command))
    log = open(log_path, 'a', encoding='utf-8') if log_path else None
    try:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            if log:
                log.write(line)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if log:
            log.close()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


if not REPO_DIR.exists():
    run_argv(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR])
run_argv(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_COMMIT or REPO_REF])
run_argv(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT or 'FETCH_HEAD'])
REPO_COMMIT_RESOLVED = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, text=True,
                                      capture_output=True).stdout.strip()
if REPO_COMMIT:
    assert REPO_COMMIT_RESOLVED == REPO_COMMIT, (REPO_COMMIT_RESOLVED, REPO_COMMIT)
NEURAL_DIR = REPO_DIR / 'final' / 'code' / 'neural_only_experiments'
for required in ('train_neural_only.py', 'test_neural_only.py', 'hand_geometry.py', 'ensemble_neural_only.py'):
    assert (NEURAL_DIR / required).is_file(), f'{required} chưa có trên nhánh {REPO_REF}: hãy commit và push code mới trước.'
if (REPO_DIR / 'requirements.txt').is_file():
    run_argv([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
print('Repository revision:', REPO_COMMIT_RESOLVED)


## 2. Kiểm tra dữ liệu trên Drive

Manifest đúng số dòng, đủ 13 ID và không giao nhau; cache tensor T64 có mặt. Không đọc test ở bước này ngoài việc đếm dòng.

In [ ]:
def read_manifest(path):
    rows = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        if line.strip():
            source, _, label = line.rsplit(maxsplit=2)
            rows.append((source, int(label)))
    return rows


manifests = {name: read_manifest(path) for name, path in (('train', TRAIN_FILE), ('val', VAL_FILE), ('test', TEST_FILE))}
for name, rows in manifests.items():
    ids = {label for _, label in rows}
    if EXPECTED_COUNTS:
        assert len(rows) == EXPECTED_COUNTS[name], (name, len(rows))
    assert ids == set(range(NUM_CLASSES)), (name, sorted(ids))
sources = {name: {source.replace(chr(92), '/') for source, _ in rows} for name, rows in manifests.items()}
assert not (sources['train'] & sources['val']) and not (sources['train'] & sources['test']) and not (sources['val'] & sources['test'])

CACHE_ROOT = None
for candidate in (DRIVE_DATA_ROOT / 'keypoint_tensor_cache', DRIVE_DATA_ROOT / 'data' / 'keypoint_tensor_cache'):
    if candidate.is_dir():
        CACHE_ROOT = candidate
        break
assert CACHE_ROOT is not None, 'Không thấy keypoint_tensor_cache trên Drive.'
print({'counts': {name: len(rows) for name, rows in manifests.items()}, 'cache_root': str(CACHE_ROOT)})


## 3. Huấn luyện 4 biến thể × các seed

Cấu hình gốc là recipe của neural C (d192, 4 lớp, 8 head, masked_mean, vận tốc, không frame masking). Mỗi run lưu trong thư mục riêng trên Drive; run đã hoàn tất được dùng lại sau khi reconnect, run dở dang được đổi tên (không xóa).

In [ ]:
BASE_ARGS = [
    '--backbone', 'transformer', '--num-classes', NUM_CLASSES, '--num-frames', 64, '--batch-size', 64,
    '--epochs', 80, '--d-model', 192, '--layers', 4, '--heads', 8, '--dropout', 0.15,
    '--label-smoothing', 0.05, '--pooling', 'masked_mean', '--class-weight-power', 0.5,
    '--jitter-std', 0.005, '--frame-mask-probability', 0.0, '--keypoint-dropout-probability', 0.02,
    '--select-metric', 'macro_f1', '--early-stop-patience', 12, '--num-workers', 0, '--log-every', 20,
]
VARIANTS = {
    'c_ref': [],
    'hand': ['--hand-features'],
    'aug': ['--time-crop-min', 0.7, '--scale-jitter', 0.1, '--shift-std', 0.03],
    'hand_aug': ['--hand-features', '--time-crop-min', 0.7, '--scale-jitter', 0.1, '--shift-std', 0.03],
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIRS = {}
for variant, extra in VARIANTS.items():
    for seed in SEEDS:
        run_dir = OUTPUT_ROOT / f'{variant}_seed{seed}'
        RUN_DIRS[(variant, seed)] = run_dir
        marker = run_dir / 'SWEEP_RUN_COMPLETE.json'
        if marker.is_file() and (run_dir / 'best.pt').is_file():
            print('Dùng lại run đã hoàn tất:', run_dir.name)
            continue
        if run_dir.exists() and any(run_dir.iterdir()):
            stale = run_dir.with_name(run_dir.name + '_incomplete_' + dt.datetime.now().strftime('%Y%m%dT%H%M%S'))
            run_dir.rename(stale)
            print('Run dở dang được đổi tên thành', stale.name)
        command = [sys.executable, '-u', NEURAL_DIR / 'train_neural_only.py',
                   '--cache-root', CACHE_ROOT, '--train-file', TRAIN_FILE, '--val-file', VAL_FILE,
                   '--output-dir', run_dir, '--seed', seed, '--device', DEVICE] + BASE_ARGS + extra
        assert '--test-file' not in command
        run_dir.mkdir(parents=True)
        run_argv(command, cwd=NEURAL_DIR, log_path=run_dir / 'train.log')
        assert (run_dir / 'best.pt').is_file()
        marker.write_text(json.dumps({'variant': variant, 'seed': seed, 'commit': REPO_COMMIT_RESOLVED,
                                      'command': [str(v) for v in command]}, indent=2), encoding='utf-8')
print('Xong huấn luyện:', len(RUN_DIRS), 'run')


## 4. Đánh giá trên validation: từng seed và ensemble

Ensemble là trung bình xác suất softmax của các seed cùng biến thể. Chỉ dùng validation ở bước này.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


val_metrics = {}
for (variant, seed), run_dir in RUN_DIRS.items():
    metrics_path = run_dir / 'val_eval' / 'metrics.json'
    if not metrics_path.is_file():
        run_argv([sys.executable, '-u', NEURAL_DIR / 'test_neural_only.py', '--checkpoint', run_dir / 'best.pt',
                  '--cache-root', CACHE_ROOT, '--test-file', VAL_FILE, '--split', 'val', '--batch-size', 128,
                  '--num-workers', 0, '--device', DEVICE, '--timing-warmup-batches', 1, '--timing-repeats', 1,
                  '--output', metrics_path], cwd=NEURAL_DIR)
    val_metrics[(variant, seed)] = read_json(metrics_path)

ENSEMBLE_ROOT = OUTPUT_ROOT / 'ensembles'
ensemble_val = {}
for variant in VARIANTS:
    out = ENSEMBLE_ROOT / variant / 'val' / 'metrics.json'
    if not out.is_file():
        run_argv([sys.executable, '-u', NEURAL_DIR / 'ensemble_neural_only.py', '--checkpoints',
                  *[RUN_DIRS[(variant, seed)] / 'best.pt' for seed in SEEDS],
                  '--cache-root', CACHE_ROOT, '--test-file', VAL_FILE, '--split', 'val', '--device', DEVICE,
                  '--output', out], cwd=NEURAL_DIR)
    ensemble_val[variant] = read_json(out)
print('Đánh giá validation xong.')


## 5. Bảng kết quả validation và chọn biến thể

Tiêu chí chọn đặt trước: **macro-F1 validation trung bình qua các seed**. Cột "ensemble" chỉ để tham khảo mức tăng nhờ trộn seed.

In [ ]:
def stats(values):
    return statistics.mean(values), (statistics.stdev(values) if len(values) > 1 else 0.0)


summary = []
for variant in VARIANTS:
    accs = [val_metrics[(variant, s)]['accuracy'] for s in SEEDS]
    f1s = [val_metrics[(variant, s)]['macro_f1'] for s in SEEDS]
    min_f1s = [min(row['f1'] for row in val_metrics[(variant, s)]['per_class'] if row['support'] > 0) for s in SEEDS]
    ens = ensemble_val[variant]
    row = {
        'variant': variant,
        'acc_mean': stats(accs)[0], 'acc_std': stats(accs)[1],
        'f1_mean': stats(f1s)[0], 'f1_std': stats(f1s)[1],
        'min_class_f1_mean': stats(min_f1s)[0],
        'ensemble_acc': ens['accuracy'], 'ensemble_f1': ens['macro_f1'],
    }
    summary.append(row)

print(f"{'variant':10} {'acc mean±sd':>16} {'macroF1 mean±sd':>18} {'min-class F1':>13} {'ens acc':>8} {'ens F1':>8}")
for row in summary:
    print(f"{row['variant']:10} {row['acc_mean']:.4f}±{row['acc_std']:.4f}   {row['f1_mean']:.4f}±{row['f1_std']:.4f}   "
          f"{row['min_class_f1_mean']:13.4f} {row['ensemble_acc']:8.4f} {row['ensemble_f1']:8.4f}")

WINNER = max(summary, key=lambda row: row['f1_mean'])['variant']
reference = next(row for row in summary if row['variant'] == 'c_ref')
winner_row = next(row for row in summary if row['variant'] == WINNER)
print('\nBiến thể thắng theo macro-F1 validation trung bình:', WINNER)
print(f"  so với c_ref: macro-F1 {winner_row['f1_mean'] - reference['f1_mean']:+.4f}, accuracy {winner_row['acc_mean'] - reference['acc_mean']:+.4f}")
noise = max(row['f1_std'] for row in summary)
if winner_row['f1_mean'] - reference['f1_mean'] < noise:
    print(f'  CẢNH BÁO: chênh lệch nhỏ hơn độ lệch chuẩn giữa các seed ({noise:.4f}); chưa kết luận được biến thể tốt hơn c_ref.')

print('\nF1 từng lớp (ensemble validation):')
names = [row['label_name'] for row in ensemble_val['c_ref']['per_class']]
print(f"{'class':6}" + ''.join(f'{v:>10}' for v in VARIANTS))
for index, name in enumerate(names):
    print(f'{name:6}' + ''.join(f"{ensemble_val[v]['per_class'][index]['f1']:10.3f}" for v in VARIANTS))


## 6. Test (khóa mặc định)

Chỉ chạy khi đặt `RUN_LOCKED_TEST=True`. Đánh giá ensemble của biến thể thắng và của `c_ref` trên test, để biết chênh lệch cùng điều kiện. **Đây là lần xem test thứ hai, nên chỉ là bằng chứng exploratory, không phải holdout sạch.** Không đổi cấu hình rồi chạy lại.

In [ ]:
test_results = {}
if RUN_LOCKED_TEST:
    for variant in dict.fromkeys([WINNER, 'c_ref']):
        out = ENSEMBLE_ROOT / variant / 'test' / 'metrics.json'
        assert not out.exists(), f'Đã có kết quả test cho {variant}; không chạy lại để tránh chọn theo test.'
        run_argv([sys.executable, '-u', NEURAL_DIR / 'ensemble_neural_only.py', '--checkpoints',
                  *[RUN_DIRS[(variant, seed)] / 'best.pt' for seed in SEEDS],
                  '--cache-root', CACHE_ROOT, '--test-file', TEST_FILE, '--split', 'test', '--device', DEVICE,
                  '--output', out], cwd=NEURAL_DIR)
        test_results[variant] = read_json(out)
    print(f"{'model':12} {'test acc':>9} {'test macroF1':>13}")
    for variant, metrics in test_results.items():
        print(f"{variant + ' (ens)':12} {metrics['accuracy']:9.4f} {metrics['macro_f1']:13.4f}")
        for member in metrics['members']:
            print(f"   seed {member['seed']}: acc={member['accuracy']:.4f} macroF1={member['macro_f1']:.4f}")
else:
    print('RUN_LOCKED_TEST=False: test chưa được chạy.')


## 7. Đóng gói kết quả vào một ZIP duy nhất

Gồm bảng tóm tắt, metric validation (và test nếu đã bật), lệnh đã chạy, hash, và checkpoint của biến thể thắng cùng `c_ref`. Từ chối ghi đè.

In [ ]:
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
run_dir_drive = DRIVE_RUN_ROOT / RUN_ID
run_zip = DRIVE_RUN_ROOT / f'{RUN_ID}.zip'
if run_dir_drive.exists() or run_zip.exists():
    raise FileExistsError(f'Từ chối ghi đè: {run_dir_drive} hoặc {run_zip}')
payload = run_dir_drive / 'payload'
payload.mkdir(parents=True)

(payload / 'summary_validation.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
(payload / 'commands.json').write_text(json.dumps(COMMANDS, indent=2), encoding='utf-8')
for variant in VARIANTS:
    target = payload / 'validation' / variant
    target.mkdir(parents=True)
    shutil.copy2(ENSEMBLE_ROOT / variant / 'val' / 'metrics.json', target / 'ensemble_val_metrics.json')
    for seed in SEEDS:
        shutil.copy2(RUN_DIRS[(variant, seed)] / 'val_eval' / 'metrics.json', target / f'seed{seed}_val_metrics.json')
        shutil.copy2(RUN_DIRS[(variant, seed)] / 'history.json', target / f'seed{seed}_history.json')
for variant, metrics in test_results.items():
    target = payload / 'test' / variant
    target.mkdir(parents=True)
    shutil.copy2(ENSEMBLE_ROOT / variant / 'test' / 'metrics.json', target / 'ensemble_test_metrics.json')
    shutil.copy2(ENSEMBLE_ROOT / variant / 'test' / 'ensemble_predictions.csv', target / 'ensemble_test_predictions.csv')
hashes = {'repo_commit': REPO_COMMIT_RESOLVED, 'manifests': {name: sha256_file(path) for name, path in
          (('train', TRAIN_FILE), ('val', VAL_FILE), ('test', TEST_FILE))}, 'checkpoints': {}}
for variant in dict.fromkeys([WINNER, 'c_ref']):
    for seed in SEEDS:
        source = RUN_DIRS[(variant, seed)] / 'best.pt'
        target = payload / 'checkpoints' / f'{variant}_seed{seed}_best.pt'
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        hashes['checkpoints'][target.name] = sha256_file(source)
(payload / 'hashes.json').write_text(json.dumps(hashes, indent=2), encoding='utf-8')
(run_dir_drive / 'run_manifest.json').write_text(json.dumps({
    'run_id': RUN_ID, 'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(), 'repo_commit': REPO_COMMIT_RESOLVED,
    'winner': WINNER, 'seeds': SEEDS, 'locked_test': RUN_LOCKED_TEST,
    'selection_rule': 'highest mean validation macro-F1 across seeds',
    'note': 'Test, if present, is the second look at the IPN test split and is exploratory.'}, indent=2), encoding='utf-8')
with zipfile.ZipFile(run_zip, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir_drive.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(run_dir_drive))
print('Run ZIP:', run_zip)
